# 自主智能体完整教程 (Autonomous Agent)

## 学习目标

本教程将整合所有组件，构建完整的自主智能体：

1. **自我反思机制** - 从经验中学习
2. **Agent 执行循环** - OODA 决策模型
3. **AutonomousAgent** - 完整系统集成
4. **实战案例** - 端到端演示

---

In [ ]:
# 环境设置
import sys
sys.path.insert(0, '../src')

from goal_manager import Goal, GoalStatus, GoalPriority, GoalManager
from action_executor import Action, ActionType, ActionExecutor, ToolAction
from self_reflection import (
    SelfReflector, SuccessAnalyzer, FailureAnalyzer,
    StrategyAdjuster, LearningMemory, ReflectionType
)
from agent_loop import LoopConfig, SimpleAgentLoop, TerminationReason
from autonomous_agent import (
    AutonomousAgent, AgentConfig, AgentBuilder, create_autonomous_agent
)

## 1. 自我反思机制

### 1.1 成功分析

In [ ]:
success_analyzer = SuccessAnalyzer()

context = {
    "goal": "实现用户登录功能",
    "actions": ["设计数据库", "编写API", "测试"],
}
outcome = "登录功能已完成，所有测试通过"

reflection = success_analyzer.reflect(context, outcome)

print(f"反思类型: {reflection.reflection_type.value}")
print(f"置信度: {reflection.confidence:.2f}")
print(f"内容: {reflection.content[:100]}...")

### 1.2 失败分析

In [ ]:
failure_analyzer = FailureAnalyzer()

context = {
    "goal": "连接数据库",
    "actions": ["建立连接"],
    "error": "Connection timeout after 30 seconds"
}
outcome = "无法连接到数据库服务器"

reflection = failure_analyzer.reflect(context, outcome)

print(f"反思类型: {reflection.reflection_type.value}")
print(f"根因: {reflection.metadata.get('root_cause', 'unknown')}")
print(f"改进建议:")
for suggestion in reflection.suggested_improvements[:3]:
    print(f"  - {suggestion}")

### 1.3 策略调整 (UCB1 算法)

$$\text{UCB}(s) = \bar{r}_s + c \sqrt{\frac{\ln N}{n_s}}$$

In [ ]:
adjuster = StrategyAdjuster(exploration_constant=1.41)

# 注册策略
adjuster.register_strategy("aggressive", "快速但可能出错")
adjuster.register_strategy("conservative", "稳健但较慢")
adjuster.register_strategy("balanced", "平衡速度与准确性")

# 模拟策略选择和结果
import random
random.seed(42)

for i in range(10):
    strategy = adjuster.select_strategy()
    # 模拟不同策略的奖励
    rewards = {"aggressive": 0.6, "conservative": 0.8, "balanced": 0.75}
    reward = rewards[strategy] + random.uniform(-0.1, 0.1)
    adjuster.record_outcome(strategy, reward)

# 查看统计
stats = adjuster.get_strategy_stats()
print("策略统计:")
for name, s in stats.items():
    print(f"  {name}: 选择{s['selections']}次, 平均奖励={s['avg_reward']:.3f}")

### 1.4 学习记忆

In [ ]:
memory = LearningMemory(max_entries=100)

# 添加经验
experiences = [
    ("处理JSON数据", "使用json.loads", "成功解析", 1.0),
    ("处理JSON数据", "手动解析", "出错", 0.2),
    ("发送HTTP请求", "使用requests库", "成功", 0.9),
    ("发送HTTP请求", "使用urllib", "成功但复杂", 0.6),
]

for sit, act, out, reward in experiences:
    memory.add_entry(sit, act, out, reward)

# 查询最佳动作
best = memory.get_best_action("处理JSON格式")
if best:
    print(f"推荐动作: {best[0]} (奖励: {best[1]:.2f})")

## 2. Agent 执行循环

### 2.1 OODA 循环配置

In [ ]:
config = LoopConfig(
    max_iterations=10,
    max_time_seconds=60,
    stuck_threshold=3,
    enable_reflection=True,
    pause_between_iterations=0,
)

print("循环配置:")
print(f"  最大迭代: {config.max_iterations}")
print(f"  超时时间: {config.max_time_seconds}秒")
print(f"  卡住阈值: {config.stuck_threshold}次连续失败")

### 2.2 简单循环演示

In [ ]:
# 模拟任务执行
task_progress = [0]

def act_function(decision):
    """执行动作并返回结果"""
    task_progress[0] += 1
    success = task_progress[0] <= 5
    return (success, f"步骤 {task_progress[0]} {'完成' if success else '失败'}")

def goal_check():
    """检查目标是否达成"""
    return task_progress[0] >= 5

# 创建并运行循环
loop = SimpleAgentLoop(
    config=LoopConfig(max_iterations=10, pause_between_iterations=0),
    act_fn=act_function,
    goal_check_fn=goal_check,
)

context = loop.run()

print(f"\n循环结束:")
print(f"  迭代次数: {context.iteration}")
print(f"  终止原因: {context.termination_reason.value if context.termination_reason else 'None'}")
print(f"  成功次数: {context.total_successes}")

## 3. AutonomousAgent 完整系统

### 3.1 基本创建

In [ ]:
# 方式1: 直接创建
agent = AutonomousAgent(AgentConfig(
    name="ResearchBot",
    max_iterations=20,
))

print(f"Agent: {agent.config.name}")
print(f"最大迭代: {agent.config.max_iterations}")

In [ ]:
# 方式2: 使用 Builder 模式
agent = (
    AgentBuilder("DataAnalyst")
    .with_config(max_iterations=30)
    .with_constraint("使用 pandas 处理数据")
    .with_constraint("结果需要可视化")
    .build()
)

print(f"Agent: {agent.config.name}")
print(f"约束: {agent.config.constraints}")

### 3.2 设置目标

In [ ]:
agent = AutonomousAgent()

# 设置主目标
goal = agent.set_objective(
    "分析销售数据并生成报告",
    priority=GoalPriority.HIGH
)

print(f"目标: {goal.description}")
print(f"优先级: {goal.priority.name}")

# 查看状态
status = agent.get_status()
print(f"\n当前状态:")
print(f"  目标总数: {status['goals']['total']}")
print(f"  Agent 状态: {status['state']}")

### 3.3 注册工具

In [ ]:
# 定义工具函数
def load_data(filename):
    return f"已加载数据: {filename}"

def analyze(data):
    return f"分析结果: 发现3个关键趋势"

def generate_report(analysis):
    return f"报告已生成: report.pdf"

# 注册工具
agent.register_tool("load_data", load_data)
agent.register_tool("analyze", analyze)
agent.register_tool("generate_report", generate_report)

print("已注册工具: load_data, analyze, generate_report")

### 3.4 查看进度

In [ ]:
progress = agent.get_progress()
print("目标进度:")
for key, value in progress.items():
    print(f"  {key}: {value}")

## 4. 综合反思器

In [ ]:
reflector = SelfReflector()

# 记录成功
r1 = reflector.reflect_on_action(
    goal="加载数据",
    action="read_csv",
    result="成功加载 1000 行数据",
    success=True
)

# 记录失败
r2 = reflector.reflect_on_action(
    goal="连接API",
    action="http_request",
    result="连接超时",
    success=False
)

# 获取洞察摘要
summary = reflector.get_insights_summary()
print("反思摘要:")
print(f"  总反思数: {summary['total_reflections']}")
print(f"  成功: {summary['successes']}")
print(f"  失败: {summary['failures']}")
print(f"  成功率: {summary['success_rate']:.1%}")

## 5. 实战案例: 研究助手

### 5.1 创建研究助手 Agent

In [ ]:
# 模拟研究工具
def search_papers(query):
    return [f"论文1: {query}相关研究", f"论文2: {query}最新进展"]

def read_paper(title):
    return f"摘要: {title}的主要贡献是..."

def summarize(texts):
    return f"总结: 共分析了{len(texts)}篇论文，发现3个主要趋势"

# 创建 Agent
research_agent = (
    AgentBuilder("ResearchAssistant")
    .with_config(max_iterations=15)
    .with_tool("search", search_papers)
    .with_tool("read", read_paper)
    .with_tool("summarize", summarize)
    .with_constraint("引用来源")
    .with_constraint("使用学术语言")
    .with_objective("研究大语言模型的最新进展")
    .build()
)

print(f"创建 Agent: {research_agent.config.name}")
print(f"目标: {research_agent.goal_manager.get_progress()['total']} 个")

### 5.2 模拟执行流程

In [ ]:
# 模拟 Agent 执行步骤
print("=== Agent 执行流程 ===")
print()

# 步骤1: 搜索论文
print("[步骤1] 搜索相关论文")
papers = search_papers("大语言模型")
for p in papers:
    print(f"  找到: {p}")

# 步骤2: 阅读论文
print("\n[步骤2] 阅读论文")
summaries = []
for p in papers:
    summary = read_paper(p)
    summaries.append(summary)
    print(f"  {summary[:50]}...")

# 步骤3: 生成总结
print("\n[步骤3] 生成研究总结")
final = summarize(summaries)
print(f"  {final}")

print("\n=== 执行完成 ===")

## 6. 练习

### 练习 1: 创建代码助手 Agent

In [ ]:
# 你的代码: 创建一个能够编写、测试、优化代码的 Agent
# ...

### 练习 2: 实现自定义反思策略

In [ ]:
# 你的代码: 实现一个基于历史成功率动态调整策略的反思器
# ...

## 总结

本教程介绍了自主智能体的完整架构：

| 组件 | 功能 | 关键类 |
|:-----|:-----|:-------|
| 目标管理 | 目标分解与调度 | `GoalManager` |
| 动作执行 | 工具调用与代码执行 | `ActionExecutor` |
| 自我反思 | 经验学习与策略调整 | `SelfReflector` |
| 执行循环 | OODA 决策循环 | `AgentLoop` |
| 自主Agent | 系统集成 | `AutonomousAgent` |

### 核心公式

- **UCB1 策略选择**: $\text{UCB}(s) = \bar{r}_s + c \sqrt{\frac{\ln N}{n_s}}$
- **指数退避**: $\text{delay}(n) = \text{base} \times 2^n$
- **目标优先级**: $\text{score} = \alpha \cdot \text{urgency} + \beta \cdot \text{importance}$